# 04 · Patrones de contaminación (no supervisado)

**Objetivo.** Identificar patrones multiparamétricos sin usar etiquetas:
K-Means, mezcla gaussiana (GMM), PCA para visualizar y Isolation Forest para
los focos anómalos.

> Requiere `python run_pipeline.py` completo (sin `--solo-datos`).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "reports" / "tables"
FIGURES = ROOT / "reports" / "figures"

In [ ]:
riesgo = pd.read_csv(PROCESSED / "dataset_riesgo.csv", encoding="utf-8-sig")
patrones = pd.read_csv(PROCESSED / "patrones.csv", encoding="utf-8-sig")
datos = riesgo.join(patrones, how="inner")
print(datos.shape)
datos[["PC1", "PC2", "cluster_kmeans", "cluster_gmm", "anomalia_score", "es_anomalia"]].head()

## 1. Elección de k

In [ ]:
import json

resumen = json.loads((TABLES / "resumen_patrones.json").read_text(encoding="utf-8"))
print("k K-Means (silueta):", resumen["k_kmeans"])
print("k GMM (BIC):", resumen["k_gmm"])
print("Varianza explicada PC1+PC2:", round(100 * resumen["varianza_acumulada_2PC"], 1), "%")
print("Umbral de anomalía (p95):", round(resumen["umbral_anomalia_p95"], 4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
km = pd.read_csv(TABLES / "seleccion_k_kmeans.csv", encoding="utf-8-sig")
gmm = pd.read_csv(TABLES / "seleccion_k_gmm.csv", encoding="utf-8-sig")
axes[0].plot(km["k"], km["silueta"], "o-", color="#2c7bb6")
axes[0].set_title("K-Means: silueta por k")
axes[0].set_xlabel("k")
axes[1].plot(gmm["k"], gmm["bic"], "o-", color="#d7191c")
axes[1].set_title("GMM: BIC por k")
axes[1].set_xlabel("k")
plt.tight_layout()
plt.show()

## 2. Clusters en el plano PCA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, col, titulo in zip(axes, ["cluster_kmeans", "cluster_gmm"], ["K-Means", "GMM"]):
    sc = ax.scatter(datos["PC1"], datos["PC2"], c=datos[col], cmap="tab10", s=26,
                    edgecolor="k", linewidth=0.2)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title(titulo)
    fig.colorbar(sc, ax=ax, label="cluster")
plt.tight_layout()
plt.show()

## 3. Perfil químico de cada cluster

In [ ]:
perfil = pd.read_csv(TABLES / "perfil_clusters.csv", encoding="utf-8-sig")
perfil

In [ ]:
from src import plots
plots.calor_perfil(riesgo, resumen["variables"],
                   datos.loc[riesgo.index, "cluster_kmeans"].to_numpy())
from IPython.display import Image, display
display(Image(filename=str(FIGURES / "11_perfil_clusters.png")))

**Lectura.** Al clustering entran solo variables químicas (se excluyen
coordenadas, año, mes y zona UTM para que la distancia mida composición y no
posición). Los perfiles resultantes son interpretables:

- un grupo **litogénico** dominado por Mn, Ti, Mg y Ca (fondo natural),
- un grupo **polimetálico** Zn-Cu-Cd-Ni (sulfuros),
- un grupo **Pb-As-Fe-Ba** (influencia minera),
- un grupo **B-Hg-Na-Cu** (firma evaporítica con anomalía de mercurio),
- grupos de **Sb-Pb-As-Mo** y **Sn-Tl-Ti-Sb** con elevaciones moderadas.

La tabla `reports/tables/perfil_clusters_z.csv` contiene la matriz completa de
z-scores por cluster para el capítulo de resultados.

## 4. Focos anómalos

In [ ]:
print("Anomalías:", int(datos["es_anomalia"].sum()), f"({100*datos['es_anomalia'].mean():.1f} %)")
datos.loc[datos["es_anomalia"], ["PUNTO_NORMALIZADO", "DISTRITO", "PLI", "nivel_riesgo",
                                "anomalia_score"]].sort_values("anomalia_score", ascending=False).head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6))
normal = datos[~datos["es_anomalia"]]
anom = datos[datos["es_anomalia"]]
ax.scatter(normal["COORD_ESTE"], normal["COORD_NORTE"], s=22, c="#bdbdbd",
           label=f"Normal (n={len(normal)})", edgecolor="k", linewidth=0.2)
ax.scatter(anom["COORD_ESTE"], anom["COORD_NORTE"], s=70, c="#d7191c", marker="^",
           label=f"Anomalía (n={len(anom)})", edgecolor="k", linewidth=0.4)
ax.set_aspect("equal")
ax.set_xlabel("Este (UTM 19S)")
ax.set_ylabel("Norte (UTM 19S)")
ax.set_title("Focos anómalos (Isolation Forest, p95)")
ax.legend()
plt.tight_layout()
plt.show()

**Lectura.** Las anomalías no son necesariamente las de mayor PLI: Isolation
Forest detecta combinaciones raras de parámetros (por ejemplo, Hg alto con
pH básico), que son candidatas a verificación en campo.